# memory.py — Walkthrough Notebook

**Course:** INFO 290: Fundamentals of Generative AI (Spring 2026)  
**Purpose:** Walk through `src/agents/memory.py` cell by cell to understand how the agent memory stream works.

#### ``What this notebook covers``
1. The `Memory` dataclass — what a single memory looks like
2. The `MemoryStream` class — how memories are stored and retrieved
3. How `load_seeds()` pre-populates an agent's memory from YAML
4. How `pretty_print()` lets us inspect the stream during debugging

#### ``Why this matters``
The memory stream is the core of the Generative Agents architecture (Park et al., 2023).  
Every observation, decision, reflection, and conversation gets stored here.  
In Stage 2, `retrieval.py` will search this stream to find the most relevant memories at decision time.

---
### Step 1: Imports

In [ ]:
# dataclass: Python decorator that auto-generates __init__ and other boilerplate
# field: used for default values in dataclasses (not used in Memory but imported for completeness)
from dataclasses import dataclass, field

# List, Optional: type hints that make the code more readable and catch bugs early
# List[Memory] means 'a list where every element is a Memory object'
# Optional[int] means 'either an int or None'
from typing import List, Optional

# numpy: used to store the embedding vector for each memory
# embeddings are 384-dimensional arrays of floats (from all-MiniLM-L6-v2)
import numpy as np

---
### Step 2: Memory types constant

In [ ]:
# the four types of memories an agent can have — used for validation in Memory.__post_init__()
# observation:  something that happened to the agent (e.g. received an insurance letter)
# decision:     something the agent decided to do (e.g. called their Firewise group)
# reflection:   a higher-level belief synthesised from multiple memories (Stage 5)
# conversation: something said in a multi-agent dialogue (Stage 4)
MEMORY_TYPES = {"observation", "decision", "reflection", "conversation"}

print("Valid memory types:", MEMORY_TYPES)

---
### Step 3: The `Memory` dataclass

In [ ]:
# @dataclass automatically generates __init__ so we don't have to write:
#   def __init__(self, timestamp, description, importance, embedding, type): ...
# Python creates it for us based on the field declarations below
@dataclass
class Memory:
    """A single entry in an agent's memory stream."""

    # the simulation day this memory was created
    # day 0 = seed memories (before simulation starts)
    # day 14 = something that happened on day 14 of the simulation
    timestamp: int

    # the memory written in first-person natural language
    # e.g. 'I received an official letter from my insurance company informing me...'
    description: str

    # how important this memory is on a scale of 1-10
    # scored by Claude via score_importance() in client.py
    # 1 = mundane (making coffee), 10 = life-altering (losing your home)
    importance: int

    # the description converted to a 384-dimensional vector by all-MiniLM-L6-v2
    # used in Stage 2 retrieval to find memories semantically similar to a new situation
    embedding: np.ndarray

    # one of: 'observation', 'decision', 'reflection', 'conversation'
    type: str

    def __post_init__(self):
        # __post_init__ runs automatically after __init__ — used for validation
        # raises an error immediately if someone tries to create a Memory with an invalid type
        # or an importance score outside 1-10, catching bugs early
        if self.type not in MEMORY_TYPES:
            raise ValueError(f"Memory type '{self.type}' must be one of {MEMORY_TYPES}")
        if not (1 <= self.importance <= 10):
            raise ValueError(f"Importance must be 1–10, got {self.importance}")

    def to_dict(self) -> dict:
        # converts the Memory object to a plain Python dict for JSONL logging
        # the key step is .tolist() on the embedding —
        # numpy arrays are not JSON-serialisable, but Python lists are
        return {
            "timestamp": self.timestamp,
            "description": self.description,
            "importance": self.importance,
            "embedding": self.embedding.tolist(),  # numpy array → plain Python list
            "type": self.type,
        }

    def __repr__(self) -> str:
        # __repr__ controls what prints when you do print(memory) or inspect a Memory object
        # we override the default @dataclass repr because the full embedding array
        # (384 floats) would make it unreadable — so we only show the first 60 chars
        return (
            f"Memory(day={self.timestamp}, type={self.type}, "
            f"importance={self.importance}, "
            f"desc='{self.description[:60]}...')"
        )

print("Memory class defined")

---
### Step 4: Create a Memory object manually to see what it looks like

In [ ]:
# create a fake embedding — normally this comes from all-MiniLM-L6-v2 via embed() in client.py
# 384 dimensions of random floats just to demonstrate the structure
fake_embedding = np.random.rand(384).astype(np.float32)

# create a Memory object directly
test_memory = Memory(
    timestamp=0,
    description="I built a brick patio using salvaged bricks in front of the house. It took three weeks of DIY work.",
    importance=7,
    embedding=fake_embedding,
    type="observation"
)

# __repr__ kicks in here — shows a readable summary without the full embedding
print("Memory object:")
print(test_memory)

print()
print("Individual fields:")
print(f"  timestamp:   {test_memory.timestamp}")
print(f"  importance:  {test_memory.importance}")
print(f"  type:        {test_memory.type}")
print(f"  description: {test_memory.description[:60]}...")
print(f"  embedding:   shape={test_memory.embedding.shape}, dtype={test_memory.embedding.dtype}")

In [ ]:
# demonstrate to_dict() — converts to JSON-serialisable format
import json

memory_dict = test_memory.to_dict()
print("to_dict() output:")
print(f"  timestamp:   {memory_dict['timestamp']}")
print(f"  importance:  {memory_dict['importance']}")
print(f"  type:        {memory_dict['type']}")
print(f"  description: {memory_dict['description'][:60]}...")
print(f"  embedding:   type={type(memory_dict['embedding'])}, length={len(memory_dict['embedding'])}")

# this works because embedding is now a list, not a numpy array
json_string = json.dumps(memory_dict)
print(f"\njson.dumps() succeeded: {len(json_string)} chars")

In [ ]:
# demonstrate validation — __post_init__ catches bad inputs immediately
try:
    bad_memory = Memory(
        timestamp=0,
        description="test",
        importance=11,           # invalid: must be 1-10
        embedding=fake_embedding,
        type="observation"
    )
except ValueError as e:
    print(f"Caught validation error: {e}")

try:
    bad_memory = Memory(
        timestamp=0,
        description="test",
        importance=5,
        embedding=fake_embedding,
        type="thought"            # invalid: not in MEMORY_TYPES
    )
except ValueError as e:
    print(f"Caught validation error: {e}")

---
### Step 5: The `MemoryStream` class

In [ ]:
class MemoryStream:
    """
    Append-only memory stream for a single agent.
    Agents never forget — retrieval controls what surfaces at decision time.
    The stream grows throughout the simulation.
    """

    def __init__(self, agent_name: str):
        # store the agent's name so we can label output clearly
        self.agent_name = agent_name
        # the stream is just a Python list of Memory objects
        # underscore prefix (_memories) signals it's private — access via methods not directly
        self._memories: List[Memory] = []

    # ── Write ─────────────────────────────────────────────────────────────────

    def add(
        self,
        description: str,
        importance: int,
        embedding: np.ndarray,
        memory_type: str,
        timestamp: int = 0,
    ) -> Memory:
        # create a Memory object from the raw inputs and append it to the list
        # this is the only way to add memories — the stream is append-only
        # (we never delete or modify existing memories, only add new ones)
        memory = Memory(
            timestamp=timestamp,
            description=description,
            importance=importance,
            embedding=embedding,
            type=memory_type,
        )
        self._memories.append(memory)
        return memory

    # ── Read ──────────────────────────────────────────────────────────────────

    def get_all(self) -> List[Memory]:
        # returns a copy of the full list in chronological order
        # used in Stage 1 where we pass all memories to the LLM
        # list() creates a copy so external code can't accidentally modify _memories
        return list(self._memories)

    def get_recent(self, n: int) -> List[Memory]:
        # returns the n most recent memories using Python list slicing
        # [-n:] means 'start from n positions before the end'
        # e.g. get_recent(3) on a 10-memory stream returns memories 8, 9, 10
        return self._memories[-n:]

    def get_by_type(self, memory_type: str) -> List[Memory]:
        # filters memories by type using a list comprehension
        # e.g. get_by_type('reflection') returns only reflection memories
        # useful in Stage 5 when we want to inspect what beliefs an agent has formed
        return [m for m in self._memories if m.type == memory_type]

    def get_cumulative_importance(self, since_index: int = 0) -> int:
        # sums the importance scores of memories from since_index onwards
        # used by reflection.py in Stage 5 to decide when to trigger a reflection
        # e.g. if the last 5 memories have a cumulative importance > threshold, reflect
        return sum(m.importance for m in self._memories[since_index:])

    def count(self) -> int:
        # simple helper — returns the total number of memories in the stream
        return len(self._memories)

    # ── Seed ──────────────────────────────────────────────────────────────────

    def load_seeds(
        self,
        seeds: List[dict],
        client_anthropic,
        agent_seed_narrative: str,
    ):
        # import here rather than at the top to avoid circular imports
        # (memory.py imports from client.py, and if client.py imported memory.py
        # at the top level, Python would get stuck in a loop)
        from src.llm.client import embed, score_importance

        print(f"[memory] Loading {len(seeds)} seed memories for {self.agent_name}...")

        for i, seed in enumerate(seeds):
            description = seed["description"]
            memory_type = seed.get("type", "observation")  # default to observation if not specified
            importance = seed.get("importance")             # None if not hardcoded in YAML

            # if importance wasn't hardcoded in jennifer.yaml, ask Claude to score it
            # passes agent_seed_narrative so Claude scores from Jennifer's perspective
            if importance is None:
                importance = score_importance(client_anthropic, description, agent_seed_narrative)

            # embed the description locally using HuggingFace all-MiniLM-L6-v2
            # returns a 384-dim numpy array — no API call, no cost
            embedding = embed(description)

            # add the memory to the stream at timestamp=0
            # (seed memories exist before the simulation starts)
            self.add(
                description=description,
                importance=importance,
                embedding=embedding,
                memory_type=memory_type,
                timestamp=0,
            )
            print(f"  [{i+1}/{len(seeds)}] '{description[:60]}...' (importance={importance})")

        print(f"[memory] Done. Stream has {self.count()} memories.\n")

    # ── Inspect ───────────────────────────────────────────────────────────────

    def pretty_print(self, n: Optional[int] = None):
        # if n is provided, only print the last n memories
        # if n is None (default), print all memories
        memories = self._memories if n is None else self._memories[-n:]
        print(f"\n{'='*60}")
        print(f"Memory stream: {self.agent_name} ({self.count()} total memories)")
        print(f"{'='*60}")
        for i, m in enumerate(memories):
            # note: embedding is intentionally not printed —
            # it's 384 floats and would be completely unreadable
            print(f"\n[{i+1}] Day {m.timestamp} | {m.type.upper()} | Importance: {m.importance}/10")
            print(f"    {m.description}")
        print(f"{'='*60}\n")

print("MemoryStream class defined")

---
### Step 6: Demo — create a MemoryStream and use it

In [ ]:
# create an empty memory stream for Jennifer
demo_stream = MemoryStream(agent_name="Jennifer")
print(f"Empty stream: {demo_stream.count()} memories")

# manually add a few memories to show how it works
demo_stream.add(
    description="I built a brick patio using salvaged bricks in front of the house.",
    importance=7,
    embedding=np.random.rand(384).astype(np.float32),
    memory_type="observation",
    timestamp=0,
)

demo_stream.add(
    description="A neighbor spent $30,000 on a basic metal fence. The fencing cost is my biggest challenge.",
    importance=8,
    embedding=np.random.rand(384).astype(np.float32),
    memory_type="observation",
    timestamp=0,
)

demo_stream.add(
    description="I received an insurance non-renewal letter on day 14.",
    importance=9,
    embedding=np.random.rand(384).astype(np.float32),
    memory_type="observation",
    timestamp=14,
)

demo_stream.add(
    description="I decided to call my Firewise group and document my brick patio compliance work.",
    importance=8,
    embedding=np.random.rand(384).astype(np.float32),
    memory_type="decision",
    timestamp=14,
)

print(f"Stream after adding memories: {demo_stream.count()} memories")

In [ ]:
# inspect the full stream
demo_stream.pretty_print()

In [ ]:
# demo the read methods
print("get_recent(2):")
for m in demo_stream.get_recent(2):
    print(f"  {m}")

print()
print("get_by_type('decision'):")
for m in demo_stream.get_by_type("decision"):
    print(f"  {m}")

print()
print(f"get_cumulative_importance() (all): {demo_stream.get_cumulative_importance()}")
print(f"get_cumulative_importance(since_index=2) (last 2): {demo_stream.get_cumulative_importance(since_index=2)}")

---
## Summary

| Class / Method | What it does |
|---|---|
| `Memory` dataclass | Stores one memory: timestamp, description, importance, embedding, type |
| `Memory.__post_init__()` | Validates type and importance on creation |
| `Memory.to_dict()` | Converts to JSON-serialisable dict (numpy → list) |
| `Memory.__repr__()` | Human-readable summary without the full embedding array |
| `MemoryStream` | Append-only list of Memory objects for one agent |
| `MemoryStream.add()` | Creates a Memory and appends it to the stream |
| `MemoryStream.get_all()` | Returns all memories in chronological order |
| `MemoryStream.get_recent(n)` | Returns the n most recent memories |
| `MemoryStream.get_by_type()` | Filters by observation / decision / reflection / conversation |
| `MemoryStream.get_cumulative_importance()` | Sums importance scores — used by reflection.py in Stage 5 |
| `MemoryStream.load_seeds()` | Pre-loads seed memories from jennifer.yaml at simulation start |
| `MemoryStream.pretty_print()` | Prints the stream in readable format for notebook debugging |